# Stage 1: Set up your machine
> Install the tools you need to work on marisco with an AI agent, on macOS, Linux or Windows

At the end of this stage your computer has one workspace that holds marisco, its Python environment and the AI agent's tools. You can run marisco's command line tools and tests, and you can start Claude Code in the marisco folder.

Some steps need administrator rights on your computer. Ask your local IT support for help with those. When a command fails, copy the complete error message and send it to the marisco maintainer.

## What you install

| Tool | What it does | Why you need it |
|---|---|---|
| Terminal | A window where you type commands | Every tool below runs from it |
| Git | Records each change to the files of a project | marisco's history and every contribution go through Git |
| GitHub, SSH key, `gh` | GitHub hosts the marisco repository. The SSH key identifies you to GitHub. `gh` is GitHub's command line tool | You send changes as pull requests and follow work in issues |
| uv | Installs Python and Python packages into isolated environments | It builds the environment that marisco and the agent share |
| Rust | A compiler toolchain | The workspace builds two of its tools from source |
| Claude Code | The AI agent. It runs in the terminal | You work on marisco with its help |
| aai-ws | Answer.AI's workspace. It holds the tool repositories, including the [aai-coding](https://github.com/AnswerDotAI/aai-coding) harness for Claude Code, in one folder with one Python environment | aai-coding gives the agent the team's coding and writing conventions, rules for when to act and when to ask you first, and the tools it uses to read, search and edit notebooks |
| VS Code | A code editor that also opens Jupyter notebooks | You read and run marisco's notebooks |

## Before you start

You need three accounts or keys:

- A GitHub account. Send your GitHub username to the marisco maintainer, who adds you as a collaborator on the repository. Collaborators can push branches and open pull requests.
- A Claude account on a paid plan (Pro, Max, Team or Enterprise). The free plan does not include Claude Code.
- A Zotero API key for the [MARIS Zotero library](https://www.zotero.org/groups/2432820/maris/library). Ask the MARIS administrators for it. Handlers that fetch bibliographic metadata need it.

## Step 1: Open a terminal

Commands in this guide go in a terminal. Type or paste one command, press Enter, and wait for the prompt to come back before the next one.

**macOS.** Open the Terminal app from Applications > Utilities. Install Apple's command line tools, which include Git and a C compiler:

```bash
xcode-select --install
```

**Linux (Ubuntu or Debian).** Open the Terminal app. Install Git, curl and the build tools:

```bash
sudo apt update
sudo apt install build-essential git curl
```

**Windows.** Use WSL, the Windows Subsystem for Linux. WSL runs Ubuntu inside Windows. After this step you follow the Linux instructions everywhere in this guide. The aai-ws setup expects a macOS or Linux shell, so this guide does not cover native Windows.

1. Install [Windows Terminal](https://aka.ms/terminal) and [VS Code](https://code.visualstudio.com/download).
2. Open Windows Terminal with a PowerShell tab, and install WSL with Ubuntu. This needs administrator rights:

   ```powershell
   wsl --install
   ```

3. Restart Windows when asked. Open Ubuntu from the Start menu, and choose a Linux username and password.
4. In the Ubuntu terminal, run the two Linux commands above.

Keep your work in your Linux home folder (`~`, which is `/home/<username>`). Do not work under `/mnt/c/`. Tools run much more slowly on Windows folders opened from WSL.

## Step 2: Install uv and Rust

Install [uv](https://docs.astral.sh/uv/):

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Install [Rust](https://www.rust-lang.org/tools/install), and accept the default installation when asked:

```bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
```

Open a new terminal, then check that both work. Each command prints a version number:

```bash
uv --version
cargo --version
```

## Step 3: Connect to GitHub

Tell Git your name and email. Git writes them on each change you record:

```bash
git config --global user.name "Your Name"
git config --global user.email "you@example.org"
```

Create an SSH key. Press Enter to accept the default file. A passphrase is optional:

```bash
ssh-keygen -t ed25519 -C "you@example.org"
```

Show the public key, and copy the line it prints:

```bash
cat ~/.ssh/id_ed25519.pub
```

On GitHub, go to Settings > SSH and GPG keys > New SSH key, and paste the line. Then check the connection. The first time, type `yes` to trust GitHub's server. GitHub replies with a greeting that includes your username:

```bash
ssh -T git@github.com
```

Install `gh` by following the [GitHub CLI instructions](https://github.com/cli/cli#installation) for your system. On WSL, follow the Linux instructions. Then sign in, choosing GitHub.com, SSH, and the browser login:

```bash
gh auth login
```

## Step 4: Install Claude Code

On WSL, run this in the Ubuntu terminal, not in PowerShell:

```bash
curl -fsSL https://claude.ai/install.sh | bash
```

Open a new terminal and check the installation. It prints a version number:

```bash
claude --version
```

Run `claude` and follow the browser prompts to sign in. Type `/exit` to quit. If the installation fails, see [Troubleshoot installation and login](https://code.claude.com/docs/en/troubleshoot-install).

## Step 5: Install the aai-ws workspace

[aai-ws](https://github.com/AnswerDotAI/aai-ws) creates the folder `~/aai-ws`, clones its tool repositories into it, and builds one Python environment in `~/aai-ws/.venv`:

```bash
uvx --from 'fastws-cli>=0.0.14' ws-setup AnswerDotAI/aai-ws ~/aai-ws
```

Activate the environment and move into the workspace:

```bash
source ~/aai-ws/.venv/bin/activate
cd ~/aai-ws
```

The rest of the setup is a runbook that Claude Code follows. Start Claude Code in manual mode. In manual mode, Claude Code asks for your approval before each action, except for a list of everyday commands:

```bash
claude --permission-mode default
```

Ask the agent to follow `aai-coding/SETUP.md`. The agent registers its Python kernel, and installs hooks and permissions for Claude Code. This is the first time the agent changes files on your computer, so do this step with the maintainer or your IT support. `SETUP.md` is written for macOS. On Linux or WSL, tell the agent which system you use, so that it skips the macOS-only steps.

Then ask the agent to set `permissions.defaultMode` to `"default"` in `~/.claude/settings.json`. Without that setting, Claude Code starts each session in auto mode, where most actions run without asking. You learn to judge the agent's actions in manual mode, and you move to auto mode at the end of [stage 3](supervise.ipynb).

Restart Claude Code when the agent has finished. The status bar under the prompt shows `manual mode on`.

You must activate the environment in every new terminal. To do it automatically, add the `source` line to your shell startup file. That file is `~/.zshrc` on macOS and `~/.bashrc` on Linux and WSL:

On macOS:

```bash
echo 'source ~/aai-ws/.venv/bin/activate' >> ~/.zshrc
```

On Linux and WSL:

```bash
echo 'source ~/aai-ws/.venv/bin/activate' >> ~/.bashrc
```

## Step 6: Add marisco to the workspace

`ws-add` clones marisco into `~/aai-ws/marisco` and installs it into the shared environment. Claude Code's Python kernel runs in that environment. The agent can therefore import marisco:

```bash
ws-add franckalbinet/marisco
```

Turn on nbdev's merge driver for notebooks. When two branches change the same notebook, Git's default merge can leave the notebook file unreadable. The nbdev driver keeps it readable, and marks any conflict inside the cells that conflict:

```bash
cd ~/aai-ws/marisco
git config --local include.path ../.gitconfig
```

Add your Zotero API key to your shell startup file (`~/.zshrc` or `~/.bashrc`), then open a new terminal:

```bash
export ZOTERO_API_KEY=your_api_key_here
```

Never paste the key into a notebook or any other file in the repository. Anything committed to the repository becomes public.

## Step 7: Install VS Code and the Jupyter extension

Install [VS Code](https://code.visualstudio.com/download) if you have not already done so, then install its Python and Jupyter extensions. On WSL, also install the WSL extension, and open marisco from the Ubuntu terminal:

```bash
code ~/aai-ws/marisco
```

Open `nbs/index.ipynb`. When VS Code asks for a kernel, choose the Python environment at `~/aai-ws/.venv/bin/python`.

## Check your setup

Open a new terminal and run:

```bash
cd ~/aai-ws/marisco
marisco-ingest not-a-handler
nbdev-test nbs/api/geo.ipynb
```

`marisco-ingest` reports that `not-a-handler` is invalid, and lists the handlers that exist. `nbdev-test` runs the notebook's tests and prints `Success.`

Then start `claude` in `~/aai-ws/marisco` and ask: "Which marisco handlers exist, and what is the status of each?" The agent answers by reading the code. Its answer shows that it can read marisco and run Python in the workspace environment.

Stage 1 is complete when all three checks work.

## Day to day

Start each working session like this:

```bash
cd ~/aai-ws/marisco
ws-sync
claude
```

`ws-sync` pulls the latest version of every repository in the workspace, including marisco, and updates the Python packages. Run it at the start of each day.

When a command you just installed gives `command not found`, open a new terminal. Installers add their folder to the command search path, and only new terminals read it. When `marisco-ingest`, `nbdev-test` or `ws-sync` gives `command not found`, the workspace environment is not active. Run `source ~/aai-ws/.venv/bin/activate`.